In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ankurzing/aspect-based-sentiment-analysis-for-financial-news/SEntFiN-v1.1.csv


In [4]:
!pip install -q -U transformers peft bitsandbytes accelerate mlflow evaluate "torchao>=0.16.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 88.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 88.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 81.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━

In [5]:
import os
import ast
import time
import pickle
import logging
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt   # uses Jupyter's inline backend on Kaggle — plots render in the notebook
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import (
    LoraConfig, PrefixTuningConfig, TaskType,
    get_peft_model, prepare_model_for_kbit_training,
)
import mlflow
import mlflow.transformers
from mlflow.tracking import MlflowClient

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [6]:
# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
LABEL2ID = {"positive": 0, "neutral": 1, "negative": 2}
ID2LABEL = {0: "positive", 1: "neutral", 2: "negative"}
NUM_LABELS = 3

MAX_LENGTH = 128
SEED = 42
BATCH_SIZE = 32
NUM_EPOCHS = 15          # raised from 6 — QLoRA's val F1 was still climbing at epoch 6 in the last
                         # run (never plateaued), so it likely has room left. Early stopping (below)
                         # protects methods like Prefix Tuning that converge/overfit much faster.
LEARNING_RATE = 2e-4     # per spec — higher LR is standard for adapter-only fine-tuning
PATIENCE = 3

TEST_SIZE = 0.15
VAL_SIZE = 0.10

FINBERT_MODEL = "ProsusAI/finbert"
MLFLOW_EXPERIMENT = "IndianFinSentiment"
MLFLOW_MODEL_NAME = "IndianFinSentiment"

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected. 4-bit QLoRA requires a CUDA GPU — enable one in Kaggle settings.")

BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR = os.path.join(BASE_DIR, "saved_models")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# MLflow 3.x defaults to a SQLite-backed store (metrics/params live INSIDE
# mlflow.db, not as flat files) unless it finds existing file-store data.
# We pin the db to a known path — also required for the Model Registry step later.
mlflow.set_tracking_uri(f"sqlite:///{os.path.join(BASE_DIR, 'mlflow.db')}")

Using device: cuda


In [7]:
# ══════════════════════════════════════════════════════════════
# DATA LOADING — SEntFiN-v1.1 (Indian financial headlines, entity-level sentiment)
# ══════════════════════════════════════════════════════════════

def resolve_sentifin_path() -> str:
    candidates = ["/kaggle/input/datasets/ankurzing/aspect-based-sentiment-analysis-for-financial-news/SEntFiN-v1.1.csv"]
    if os.path.isdir("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            for f in files:
                if f == "SEntFiN-v1.1.csv":
                    candidates.insert(0, os.path.join(root, f))
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        "SEntFiN-v1.1.csv not found. On Kaggle: Add Data -> upload SEntFiN-v1.1.csv as a "
        "dataset and attach it to this notebook. Locally: place it at data/raw/SEntFiN-v1.1.csv."
    )


def load_sentifin() -> pd.DataFrame:
    """
    SEntFiN-v1.1 — 10,753 Indian financial headlines with entity-level sentiment,
    e.g. Decisions = {"SpiceJet": "neutral"}. ~26% of headlines mention >1 entity.
    Since this notebook classifies a whole headline as one label (not entity-aware),
    we keep headlines where every entity shares one sentiment (trivially true for
    single-entity headlines) and drop the ~11.5% where entities disagree — those are
    ambiguous for a single-label classifier.
    """
    log.info("Loading SEntFiN-v1.1...")
    raw = pd.read_csv(resolve_sentifin_path())

    texts, labels, n_entities = [], [], []
    dropped = 0
    for _, row in raw.iterrows():
        try:
            decisions = ast.literal_eval(row["Decisions"])
        except (ValueError, SyntaxError):
            continue
        if not decisions:
            continue
        unique_labels = set(decisions.values())
        if len(unique_labels) != 1:
            dropped += 1
            continue
        texts.append(row["Title"])
        labels.append(unique_labels.pop())
        n_entities.append(len(decisions))

    df = pd.DataFrame({"text": texts, "label": labels, "n_entities": n_entities})
    df["label"] = df["label"].str.lower().str.strip()
    df = df[df["label"].isin(LABEL2ID.keys())]
    df["label_id"] = df["label"].map(LABEL2ID)
    df["source"] = "sentifin"
    log.info(f"  Loaded {len(df)} headlines (dropped {dropped} with conflicting entity sentiment)")
    return df[["text", "label", "label_id", "source", "n_entities"]]


def load_and_combine() -> pd.DataFrame:
    """Load SEntFiN, clean, dedupe. Returns one clean DataFrame."""
    df = load_sentifin()

    df = df.dropna(subset=["text", "label_id"])
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"].str.len() > 10]         # drop near-empty rows
    df = df[df["text"].str.len() < 1000]       # drop unusually long rows
    df = df.drop_duplicates(subset=["text"])

    log.info(f"Combined dataset: {len(df)} samples")
    for label, count in df["label"].value_counts().items():
        log.info(f"  {label:>10}: {count:>6} ({count/len(df)*100:.1f}%)")

    return df.reset_index(drop=True)


def split_data(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Stratified train/val/test split so class balance holds in every split."""
    train_val_df, test_df = train_test_split(
        df, test_size=TEST_SIZE, stratify=df["label_id"], random_state=SEED
    )
    train_df, val_df = train_test_split(
        train_val_df, test_size=VAL_SIZE / (1 - TEST_SIZE),
        stratify=train_val_df["label_id"], random_state=SEED
    )
    log.info(f"Split sizes -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


df_all = load_and_combine()
train_df, val_df, test_df = split_data(df_all)

2026-07-17 11:31:09,456 | INFO | Loading SEntFiN-v1.1...
2026-07-17 11:31:10,117 | INFO |   Loaded 9518 headlines (dropped 1235 with conflicting entity sentiment)
2026-07-17 11:31:10,134 | INFO | Combined dataset: 9459 samples
2026-07-17 11:31:10,136 | INFO |      neutral:   3407 (36.0%)
2026-07-17 11:31:10,136 | INFO |     positive:   3352 (35.4%)
2026-07-17 11:31:10,137 | INFO |     negative:   2700 (28.5%)
2026-07-17 11:31:10,148 | INFO | Split sizes -> Train: 7094 | Val: 946 | Test: 1419


In [8]:
hf_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)

def tokenize_df(df: pd.DataFrame):
    enc = hf_tokenizer(
        df["text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    labels = torch.tensor(df["label_id"].tolist(), dtype=torch.long)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)


train_ds = tokenize_df(train_df)
val_ds = tokenize_df(val_df)
test_ds = tokenize_df(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# class weights for the imbalanced 3-class problem
class_counts = train_df["label_id"].value_counts().sort_index().values
class_weights = torch.tensor(len(train_df) / (NUM_LABELS * class_counts), dtype=torch.float32).to(DEVICE)
print(f"Class weights (positive, neutral, negative): {class_weights.tolist()}")

2026-07-17 11:31:15,034 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 11:31:15,035 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-17 11:31:15,053 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-17 11:31:15,071 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

2026-07-17 11:31:15,196 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 11:31:15,213 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-17 11:31:15,230 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

2026-07-17 11:31:15,311 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-07-17 11:31:15,377 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-07-17 11:31:15,436 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/vocab.txt "HTTP/1.1 307 Temporary Redirect"
2026-07-17 11:31:15,454 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/vocab.txt "HTTP/1.1 200 OK"
2026-07-17 11:31:15,473 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/vocab.txt "HTTP/1.1 200 OK"


vocab.txt: 0.00B [00:00, ?B/s]

2026-07-17 11:31:15,571 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/tokenizer.json "HTTP/1.1 404 Not Found"
2026-07-17 11:31:15,636 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-07-17 11:31:15,698 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 11:31:15,716 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/special_tokens_map.json "HTTP/1.1 200 OK"
2026-07-17 11:31:15,735 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

2026-07-17 11:31:15,810 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Class weights (positive, neutral, negative): [0.9405993223190308, 0.925505518913269, 1.1677366495132446]


In [10]:
def compute_metrics(y_true, y_pred) -> dict:
    per_class = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2])
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "f1_weighted": round(f1_score(y_true, y_pred, average="weighted"), 4),
        "f1_positive": round(per_class[0], 4),
        "f1_neutral": round(per_class[1], 4),
        "f1_negative": round(per_class[2], 4),
    }


def save_confusion_matrix(y_true, y_pred, model_name: str) -> str:
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_pct, annot=True, fmt=".2%", cmap="Blues",
                xticklabels=["positive", "neutral", "negative"],
                yticklabels=["positive", "neutral", "negative"], ax=ax, linewidths=0.5)
    ax.set_title(f"Confusion Matrix - {model_name}", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    path = os.path.join(RESULTS_DIR, f"cm_{model_name.lower().replace(' ', '_').replace('/', '-')}.png")
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    return path


def trainable_param_stats(model) -> dict:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return {"trainable_params": trainable, "total_params": total, "trainable_pct": round(100 * trainable / total, 4)}


def train_peft_epoch(model, loader, optimizer, criterion, scheduler=None) -> float:
    model.train()
    total_loss = 0.0
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item() * input_ids.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_peft(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        loss = criterion(logits, labels)
        total_loss += loss.item() * input_ids.size(0)

        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, compute_metrics(all_labels, all_preds), all_preds, all_labels


def fit_peft(model, model_name: str, epochs=NUM_EPOCHS, lr=LEARNING_RATE, patience=PATIENCE):
    """Same early-stopping-on-val-F1 pattern used for Models 3-6, adapted to HF's forward() signature.

    Uses a linear warmup + linear decay learning rate schedule (10% warmup) — standard for
    transformer fine-tuning, and it directly addresses what we saw in the last run: Prefix Tuning's
    val F1 peaked at epoch 1 and got steadily worse afterward under a flat learning rate, which is
    a classic sign the LR stayed too high for too long. Decaying it over training gives every
    variant a chance to settle into a better minimum instead of overshooting it.

    NOTE: the best-epoch checkpoint only snapshots/restores the TRAINABLE parameters (the
    LoRA/prefix adapter weights + classifier head), not the full model.state_dict(). For QLoRA,
    the frozen base model is 4-bit quantized via bitsandbytes, which stores extra per-weight
    buffers (absmax, quant_map, nested_absmax, ...) alongside each quantized tensor — those don't
    round-trip cleanly through state_dict()/load_state_dict() (cloning + reloading them raises
    "Unexpected key(s) in state_dict"). Since the base model is frozen and never changes during
    training, only the trainable adapter parameters ever need to be checkpointed anyway.
    """
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    total_steps = len(train_loader) * epochs
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    best_val_f1, best_state, epochs_no_improve = -1.0, None, 0
    start = time.time()

    for epoch in range(1, epochs + 1):
        train_loss = train_peft_epoch(model, train_loader, optimizer, criterion, scheduler)
        val_loss, val_metrics, _, _ = evaluate_peft(model, val_loader, criterion)

        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        for k, v in val_metrics.items():
            mlflow.log_metric(f"val_{k}", v, step=epoch)

        print(f"[{model_name}] Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | val_f1={val_metrics['f1_weighted']:.4f}")

        if val_metrics["f1_weighted"] > best_val_f1:
            best_val_f1 = val_metrics["f1_weighted"]
            best_state = {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    train_time = time.time() - start
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in best_state:
                param.copy_(best_state[name].to(param.device))
    return model, train_time, best_val_f1

In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    # Without this, the classifier head (freshly initialized for our 3-class problem,
    # not part of FinBERT's pretrained weights) also gets quantized into a non-floating-point
    # Linear4bit layer. PEFT then can't set requires_grad=True on it, and get_peft_model()
    # crashes with "only Tensors of floating point dtype can require gradients".
    llm_int8_skip_modules=["classifier"],
)

base_model_qlora = AutoModelForSequenceClassification.from_pretrained(
    FINBERT_MODEL, num_labels=NUM_LABELS, quantization_config=bnb_config, device_map={"": 0}
)
base_model_qlora = prepare_model_for_kbit_training(base_model_qlora)

qlora_config = LoraConfig(
    r=8, lora_alpha=16, target_modules=["query", "value"],
    lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_CLS,
)
qlora_model = get_peft_model(base_model_qlora, qlora_config)
qlora_model.print_trainable_parameters()

mlflow.set_experiment(MLFLOW_EXPERIMENT)
with mlflow.start_run(run_name="FinBERT-QLoRA"):
    mlflow.log_params({
        "peft_method": "QLoRA", "quant": "4bit-nf4", "r": 8, "lora_alpha": 16,
        "target_modules": "query,value", "lora_dropout": 0.1, "lr": LEARNING_RATE,
    })

    qlora_model, qlora_time, qlora_val_f1 = fit_peft(qlora_model, "QLoRA")

    test_loss, test_metrics, test_preds, test_labels = evaluate_peft(
        qlora_model, test_loader, nn.CrossEntropyLoss(weight=class_weights)
    )
    stats = trainable_param_stats(qlora_model)

    for k, v in test_metrics.items():
        mlflow.log_metric(f"test_{k}", v)
    mlflow.log_metric("train_time_seconds", round(qlora_time, 2))
    mlflow.log_metrics(stats)

    cm_path = save_confusion_matrix(test_labels, test_preds, "FinBERT-QLoRA")
    mlflow.log_artifact(cm_path)

    qlora_model.save_pretrained(os.path.join(MODELS_DIR, "finbert_qlora_adapter"))
    mlflow.transformers.log_model(
        transformers_model={"model": qlora_model, "tokenizer": hf_tokenizer},
        name="model", task="text-classification",
    )
    qlora_run_id = mlflow.active_run().info.run_id

qlora_results = {"name": "QLoRA (r=8, q+v)", "test_metrics": test_metrics, "stats": stats,
                  "train_time": qlora_time, "run_id": qlora_run_id}
print(qlora_results)

2026-07-17 11:31:42,022 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 11:31:42,040 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-17 11:31:42,108 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-07-17 11:31:42,184 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 11:31:42,199 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-17 11:31:46,291 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-07-17 11:31:46,355 

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

2026-07-17 11:31:49,906 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-07-17 11:31:50,031 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert "HTTP/1.1 200 OK"
2026-07-17 11:31:50,147 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/commits/main "HTTP/1.1 200 OK"
2026-07-17 11:31:50,248 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/discussions?p=0 "HTTP/1.1 200 OK"
2026-07-17 11:31:50,352 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/commits/refs%2Fpr%2F29 "HTTP/1.1 200 OK"
2026-07-17 11:31:50,464 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-07-17 11:31:50,550 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors "HTTP/1.1 302 Foun

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

[QLoRA] Epoch 14/15 | train_loss=0.1901 | val_loss=0.3479 | val_f1=0.8802
Early stopping at epoch 14.


2026-07-17 11:48:28,503 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 11:48:28,518 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-17 11:48:28,577 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 11:48:28,593 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026/07/17 11:48:28 INFO mlflow.transformers: Overriding save_pretrained to False for PEFT models, following the Transformers behavior. The PEFT adaptor and config will be saved, but the base model weights will not and reference to the HuggingFace Hub repository will be logged instead.
2026-07-17 11:48:28,984 | INFO | HT

README.md: 0.00B [00:00, ?B/s]

2026-07-17 11:48:29,442 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026/07/17 11:48:29 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/17 11:48:29 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/17 11:48:29 INFO mlflow.transformers: A local checkpoint path or P

{'name': 'QLoRA (r=8, q+v)', 'test_metrics': {'accuracy': 0.8633, 'f1_weighted': 0.8631, 'f1_positive': np.float64(0.8735), 'f1_neutral': np.float64(0.8446), 'f1_negative': np.float64(0.8735)}, 'stats': {'trainable_params': 297219, 'total_params': 67019526, 'trainable_pct': 0.4435}, 'train_time': 986.716552734375, 'run_id': '77bf54bb1260418392e2eca786b5585b'}


In [12]:
base_model_lora = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL, num_labels=NUM_LABELS).to(DEVICE)

lora_config = LoraConfig(
    r=16, lora_alpha=16, target_modules=["query", "key", "value"],
    lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_CLS,
)
lora_model = get_peft_model(base_model_lora, lora_config)
lora_model.print_trainable_parameters()

with mlflow.start_run(run_name="FinBERT-LoRA"):
    mlflow.log_params({
        "peft_method": "LoRA", "quant": "none", "r": 16, "lora_alpha": 16,
        "target_modules": "query,key,value", "lora_dropout": 0.1, "lr": LEARNING_RATE,
    })

    lora_model, lora_time, lora_val_f1 = fit_peft(lora_model, "LoRA")

    test_loss, test_metrics, test_preds, test_labels = evaluate_peft(
        lora_model, test_loader, nn.CrossEntropyLoss(weight=class_weights)
    )
    stats = trainable_param_stats(lora_model)

    for k, v in test_metrics.items():
        mlflow.log_metric(f"test_{k}", v)
    mlflow.log_metric("train_time_seconds", round(lora_time, 2))
    mlflow.log_metrics(stats)

    cm_path = save_confusion_matrix(test_labels, test_preds, "FinBERT-LoRA")
    mlflow.log_artifact(cm_path)

    lora_model.save_pretrained(os.path.join(MODELS_DIR, "finbert_lora_adapter"))
    mlflow.transformers.log_model(
        transformers_model={"model": lora_model, "tokenizer": hf_tokenizer},
        name="model", task="text-classification",
    )
    lora_run_id = mlflow.active_run().info.run_id

lora_results = {"name": "LoRA (r=16, q+k+v)", "test_metrics": test_metrics, "stats": stats,
                 "train_time": lora_time, "run_id": lora_run_id}
print(lora_results)

2026-07-17 11:48:56,140 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 11:48:56,156 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-17 11:48:56,219 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 11:48:56,236 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-17 11:48:56,297 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-07-17 11:48:56,408 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert "HTTP/1.1 200 OK"
2026-07-17 11:48:56,503 | INFO | HTTP Request: GET htt

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-07-17 11:48:56,838 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/commits/refs%2Fpr%2F29 "HTTP/1.1 200 OK"
2026-07-17 11:48:56,899 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-07-17 11:48:56,964 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors "HTTP/1.1 302 Found"


trainable params: 887,043 || all params: 110,371,590 || trainable%: 0.8037
[LoRA] Epoch 1/15 | train_loss=1.2821 | val_loss=0.6515 | val_f1=0.7136
[LoRA] Epoch 2/15 | train_loss=0.5098 | val_loss=0.4272 | val_f1=0.8264
[LoRA] Epoch 3/15 | train_loss=0.3910 | val_loss=0.3737 | val_f1=0.8604
[LoRA] Epoch 4/15 | train_loss=0.3295 | val_loss=0.3622 | val_f1=0.8622
[LoRA] Epoch 5/15 | train_loss=0.2967 | val_loss=0.3370 | val_f1=0.8692
[LoRA] Epoch 6/15 | train_loss=0.2683 | val_loss=0.3334 | val_f1=0.8766
[LoRA] Epoch 7/15 | train_loss=0.2459 | val_loss=0.3334 | val_f1=0.8748
[LoRA] Epoch 8/15 | train_loss=0.2190 | val_loss=0.3381 | val_f1=0.8761
[LoRA] Epoch 9/15 | train_loss=0.2121 | val_loss=0.3410 | val_f1=0.8812
[LoRA] Epoch 10/15 | train_loss=0.1962 | val_loss=0.3443 | val_f1=0.8750
[LoRA] Epoch 11/15 | train_loss=0.1776 | val_loss=0.3531 | val_f1=0.8770
[LoRA] Epoch 12/15 | train_loss=0.1681 | val_loss=0.3683 | val_f1=0.8813
[LoRA] Epoch 13/15 | train_loss=0.1574 | val_loss=0.3622 |

2026-07-17 12:16:22,053 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 12:16:22,071 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-17 12:16:22,129 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 12:16:22,144 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026/07/17 12:16:22 INFO mlflow.transformers: Overriding save_pretrained to False for PEFT models, following the Transformers behavior. The PEFT adaptor and config will be saved, but the base model weights will not and reference to the HuggingFace Hub repository will be logged instead.
2026-07-17 12:16:22,296 | INFO | HT

{'name': 'LoRA (r=16, q+k+v)', 'test_metrics': {'accuracy': 0.8654, 'f1_weighted': 0.8652, 'f1_positive': np.float64(0.8788), 'f1_neutral': np.float64(0.8441), 'f1_negative': np.float64(0.875)}, 'stats': {'trainable_params': 887043, 'total_params': 110371590, 'trainable_pct': 0.8037}, 'train_time': 1634.8522238731384, 'run_id': '8c5575db1bfc41f69e4ef99ecfb3fe0b'}


## 9. Hyperparameter tuning — LoRA (best variant so far: 86.54% test accuracy / 0.8652 F1)

**Why LoRA, not QLoRA/Prefix Tuning:** LoRA already beat both on this data (and on the old
noisy dataset too), so the tuning budget below goes entirely into pushing LoRA further rather
than re-tuning three variants in parallel.

**Why `max_length` is now a real tuning axis, not a fixed constant:** the run above used
`MAX_LENGTH=128`, inherited from the old dataset which included long `financial_phrasebank`
sentences. SEntFiN headlines are much shorter — median 16 FinBERT tokens, 99th percentile 27,
max 38. At 128, roughly 80-90% of every sequence is pure `[PAD]` — wasted compute with zero
signal. 32-64 should be enough to cover virtually every headline while training several times
faster per epoch.

**Why the search is random, not an exhaustive grid:** the space below (learning rate x LoRA
rank x alpha x target modules x dropout x max_length x batch size x weight decay x optimizer x
scheduler x warmup x label smoothing x layer-unfreezing) has 100,000+ combinations — a full grid
is not feasible on a single Kaggle GPU session. Random search over a bounded trial budget is the
standard, well-justified alternative (Bergstra & Bengio, 2012): it explores the space far more
efficiently per trial than grid search when only a few dimensions actually matter.

**Why some knobs specifically target overfitting:** the LoRA run above shows a classic
overfitting signature — train_loss fell monotonically to 0.147 by epoch 15, while val_loss
bottomed out around 0.333-0.334 at epochs 6-7 and drifted back up to 0.364 by epoch 15 (val F1
kept inching up anyway because F1 and loss don't move in lockstep, but the gap is a real
regularization gap). `lora_dropout`, `weight_decay`, `label_smoothing`, and smaller `lora_r`
are all in the search space specifically to close that gap.

**Two-stage plan:**
1. **Search stage** — `N_TRIALS` random configs, each trained for a short budget
   (`SEARCH_EPOCHS`, aggressive `SEARCH_PATIENCE`) just to *rank* configs by val F1. Not meant to
   fully converge any of them.
2. **Final stage** — retrain only the winning config for the full epoch budget, evaluate the
   test set **once** (repeatedly checking test performance across many configs would leak
   information from the held-out set into model selection).

**New technique — partial layer unfreezing:** on top of *which* attention projections get a LoRA
adapter (`target_modules`), `unfreeze_last_n_layers` additionally makes the last 1-2 FinBERT
transformer blocks fully trainable (not just LoRA-adapted) — the layers closest to the
classification head, where task-specific features matter most. This is a distinct axis from
`target_modules` and directly answers "tune different layers," not just which projection
matrices inside every layer get adapted.

Rough compute budget: the untuned LoRA run above took ~27 min for up to 15 epochs at
`max_length=128, batch_size=32`. At `max_length` in [32, 48, 64] each epoch should run
roughly 2-4x faster, so `N_TRIALS=12` short trials (`SEARCH_EPOCHS=5`) should take well under an
hour, leaving plenty of session budget for the final full retrain. Raise `N_TRIALS` if you want a
more thorough search — cost scales linearly.

In [ ]:
# ══════════════════════════════════════════════════════════════
# SEARCH SPACE + HELPERS
# ══════════════════════════════════════════════════════════════
import random
from transformers import get_cosine_schedule_with_warmup

SEARCH_SEED = 123

SEARCH_SPACE = {
    "learning_rate": [1e-4, 2e-4, 3e-4, 5e-4],
    "lora_r": [8, 16, 32],
    "lora_alpha_mult": [1, 2],                 # lora_alpha = lora_r * this
    "target_modules": [
        ("query", "value"),
        ("query", "key", "value"),
        ("query", "key", "value", "dense"),
    ],
    "lora_dropout": [0.05, 0.1, 0.2],
    "max_length": [32, 48, 64],                 # was a fixed 128 — see markdown above
    "batch_size": [16, 32, 64],
    "weight_decay": [0.0, 0.01, 0.1],
    "optimizer_name": ["adamw", "adam", "rmsprop"],
    "scheduler_type": ["linear", "cosine"],
    "warmup_ratio": [0.06, 0.1],
    "label_smoothing": [0.0, 0.05, 0.1],
    "unfreeze_last_n_layers": [0, 1, 2],        # 0 = pure LoRA, like the baseline run above
}


def sample_config(rng: random.Random) -> dict:
    return {k: rng.choice(v) for k, v in SEARCH_SPACE.items()}


def build_loaders(max_length: int, batch_size: int):
    """Re-tokenize at a given max_length and wrap in DataLoaders at a given batch_size.
    Cheap enough to redo per trial — tokenizing ~9.5k short headlines takes well under a second."""
    def _tok(df):
        enc = hf_tokenizer(
            df["text"].tolist(), padding="max_length", truncation=True,
            max_length=max_length, return_tensors="pt",
        )
        labels = torch.tensor(df["label_id"].tolist(), dtype=torch.long)
        return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)

    tr = DataLoader(_tok(train_df), batch_size=batch_size, shuffle=True)
    va = DataLoader(_tok(val_df), batch_size=batch_size, shuffle=False)
    te = DataLoader(_tok(test_df), batch_size=batch_size, shuffle=False)
    return tr, va, te


def build_optimizer(name: str, params, lr: float, weight_decay: float):
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")


def build_scheduler(kind: str, optimizer, warmup_steps: int, total_steps: int):
    if kind == "linear":
        return get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    if kind == "cosine":
        return get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    raise ValueError(f"Unknown scheduler: {kind}")


def apply_layer_unfreezing(model, n_layers: int):
    """Fully unfreeze the last n transformer blocks (on top of whatever LoRA already adapts).
    Matches on the 'encoder.layer.{i}.' substring, which survives PEFT's module-path wrapping
    regardless of exact prefix (e.g. base_model.model.bert.encoder.layer.11...)."""
    if n_layers <= 0:
        return
    total_layers = model.config.num_hidden_layers if hasattr(model, "config") else 12
    unfreeze_from = total_layers - n_layers
    target_substrings = [f"encoder.layer.{i}." for i in range(unfreeze_from, total_layers)]
    for name, param in model.named_parameters():
        if any(sub in name for sub in target_substrings):
            param.requires_grad = True

In [ ]:
# ══════════════════════════════════════════════════════════════
# TRIAL RUNNER — trains one config, returns its best val F1 + the trained model
# ══════════════════════════════════════════════════════════════

def run_lora_trial(config: dict, epochs: int, patience: int, log_to_mlflow: bool = True):
    tr_loader, va_loader, te_loader = build_loaders(config["max_length"], config["batch_size"])

    base_model = AutoModelForSequenceClassification.from_pretrained(
        FINBERT_MODEL, num_labels=NUM_LABELS
    ).to(DEVICE)

    lora_alpha = config["lora_r"] * config["lora_alpha_mult"]
    lora_cfg = LoraConfig(
        r=config["lora_r"], lora_alpha=lora_alpha,
        target_modules=list(config["target_modules"]),
        lora_dropout=config["lora_dropout"], bias="none",
        task_type=TaskType.SEQ_CLS, modules_to_save=["classifier"],
    )
    model = get_peft_model(base_model, lora_cfg)
    apply_layer_unfreezing(model, config["unfreeze_last_n_layers"])

    optimizer = build_optimizer(
        config["optimizer_name"], [p for p in model.parameters() if p.requires_grad],
        config["learning_rate"], config["weight_decay"],
    )
    total_steps = len(tr_loader) * epochs
    warmup_steps = int(config["warmup_ratio"] * total_steps)
    scheduler = build_scheduler(config["scheduler_type"], optimizer, warmup_steps, total_steps)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=config["label_smoothing"])

    best_val_f1, best_state, epochs_no_improve = -1.0, None, 0
    start = time.time()
    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_peft_epoch(model, tr_loader, optimizer, criterion, scheduler)
        val_loss, val_metrics, _, _ = evaluate_peft(model, va_loader, criterion)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, **val_metrics})

        if log_to_mlflow:
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            for k, v in val_metrics.items():
                mlflow.log_metric(f"val_{k}", v, step=epoch)

        if val_metrics["f1_weighted"] > best_val_f1:
            best_val_f1 = val_metrics["f1_weighted"]
            best_state = {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    train_time = time.time() - start
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in best_state:
                param.copy_(best_state[name].to(param.device))

    return {
        "model": model, "best_val_f1": best_val_f1, "history": history,
        "train_time": train_time, "test_loader": te_loader, "criterion": criterion,
        "config": config,
    }

In [ ]:
# ══════════════════════════════════════════════════════════════
# STAGE 1 — random search (short budget per trial, just to RANK configs)
# ══════════════════════════════════════════════════════════════
N_TRIALS = 12          # raise for a more thorough search — cost scales linearly
SEARCH_EPOCHS = 5      # short budget: enough to see which configs are promising, not to converge
SEARCH_PATIENCE = 2

rng = random.Random(SEARCH_SEED)
trial_configs = [sample_config(rng) for _ in range(N_TRIALS)]

leaderboard = []
mlflow.set_experiment(MLFLOW_EXPERIMENT)

for i, cfg in enumerate(trial_configs, start=1):
    print(f"\n=== Trial {i}/{N_TRIALS} ===")
    print(cfg)
    with mlflow.start_run(run_name=f"LoRA-HPT-trial-{i}"):
        loggable_cfg = {**cfg, "target_modules": ",".join(cfg["target_modules"])}
        mlflow.log_params(loggable_cfg)

        result = run_lora_trial(cfg, epochs=SEARCH_EPOCHS, patience=SEARCH_PATIENCE)
        mlflow.log_metric("best_val_f1", result["best_val_f1"])

        leaderboard.append({
            "trial": i, **loggable_cfg,
            "best_val_f1": result["best_val_f1"],
            "train_time": round(result["train_time"], 1),
        })

    del result
    torch.cuda.empty_cache()

leaderboard_df = pd.DataFrame(leaderboard).sort_values("best_val_f1", ascending=False).reset_index(drop=True)
leaderboard_df.to_csv(os.path.join(RESULTS_DIR, "lora_hpt_leaderboard.csv"), index=False)
print(f"\nSearch done — total time: {leaderboard_df['train_time'].sum() / 60:.1f} min")
leaderboard_df

## 10. Stage 2 — retrain the winning config for real, evaluate test set once

In [ ]:
# ══════════════════════════════════════════════════════════════
# Pick the winning config, retrain fully, evaluate test set ONCE
# ══════════════════════════════════════════════════════════════
best_row = leaderboard_df.iloc[0]
best_config = {k: best_row[k] for k in SEARCH_SPACE.keys()}
best_config["target_modules"] = tuple(best_row["target_modules"].split(","))
# leaderboard_df stores numeric fields as numpy scalars — plain python types are safer downstream
for k in ["lora_r", "lora_alpha_mult", "max_length", "batch_size", "unfreeze_last_n_layers"]:
    best_config[k] = int(best_config[k])
for k in ["learning_rate", "lora_dropout", "weight_decay", "warmup_ratio", "label_smoothing"]:
    best_config[k] = float(best_config[k])

print("Best config from search (val F1 =", best_row["best_val_f1"], "):")
print(best_config)

FINAL_EPOCHS = 15
FINAL_PATIENCE = 5   # more patient than the original run's 3 — LoRA's val F1 was still (slowly)
                     # improving through epoch 13 last time, so 3 risked stopping right as it
                     # was still finding gains. Extra epochs are cheap once we've narrowed to
                     # a single config instead of paying this cost across every search trial.

with mlflow.start_run(run_name="FinBERT-LoRA-Tuned"):
    loggable_best = {**best_config, "target_modules": ",".join(best_config["target_modules"])}
    mlflow.log_params(loggable_best)
    mlflow.log_param("epochs_budget", FINAL_EPOCHS)
    mlflow.log_param("patience", FINAL_PATIENCE)

    final_result = run_lora_trial(best_config, epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE)
    tuned_model = final_result["model"]

    test_loss, test_metrics, test_preds, test_labels = evaluate_peft(
        tuned_model, final_result["test_loader"], final_result["criterion"]
    )
    stats = trainable_param_stats(tuned_model)

    for k, v in test_metrics.items():
        mlflow.log_metric(f"test_{k}", v)
    mlflow.log_metric("train_time_seconds", round(final_result["train_time"], 2))
    mlflow.log_metrics(stats)

    cm_path = save_confusion_matrix(test_labels, test_preds, "FinBERT-LoRA-Tuned")
    mlflow.log_artifact(cm_path)

    tuned_model.save_pretrained(os.path.join(MODELS_DIR, "finbert_lora_tuned_adapter"))
    mlflow.transformers.log_model(
        transformers_model={"model": tuned_model, "tokenizer": hf_tokenizer},
        name="model", task="text-classification",
    )
    tuned_run_id = mlflow.active_run().info.run_id

print(f"\n{'='*55}\n  FinBERT-LoRA-Tuned — TEST RESULTS\n{'='*55}")
print(f"  Accuracy    : {test_metrics['accuracy']:.4f}")
print(f"  F1 Weighted : {test_metrics['f1_weighted']:.4f}")
print(f"  F1 Positive : {test_metrics['f1_positive']:.4f}")
print(f"  F1 Neutral  : {test_metrics['f1_neutral']:.4f}")
print(f"  F1 Negative : {test_metrics['f1_negative']:.4f}")
print(f"{'='*55}")
print(classification_report(test_labels, test_preds, target_names=["positive", "neutral", "negative"]))

tuned_results = {"name": "LoRA-Tuned", "test_metrics": test_metrics, "stats": stats,
                  "train_time": final_result["train_time"], "run_id": tuned_run_id,
                  "config": best_config}
print(tuned_results)

## 11. Compare tuned vs. baseline, update the registry if it actually won

In [ ]:
print(f"{'Method':<24} {'F1 (weighted)':>14} {'Accuracy':>10}")
print("-" * 52)
print(f"{'LoRA (r=16, baseline)':<24} {lora_results['test_metrics']['f1_weighted']:>14.4f} "
      f"{lora_results['test_metrics']['accuracy']:>10.4f}")
print(f"{'LoRA-Tuned':<24} {tuned_results['test_metrics']['f1_weighted']:>14.4f} "
      f"{tuned_results['test_metrics']['accuracy']:>10.4f}")

if tuned_results["test_metrics"]["f1_weighted"] > lora_results["test_metrics"]["f1_weighted"]:
    print("\nTuned model beat the baseline — registering it as Production.")
    client = MlflowClient()
    model_uri = f"runs:/{tuned_results['run_id']}/model"
    try:
        registered = mlflow.register_model(model_uri=model_uri, name=MLFLOW_MODEL_NAME)
        client.set_registered_model_alias(name=MLFLOW_MODEL_NAME, alias="Production", version=registered.version)
        print(f"Registered tuned LoRA as '{MLFLOW_MODEL_NAME}' v{registered.version} -> alias=Production")
    except Exception as e:
        print(f"Registry step skipped/failed ({e}). Adapter files are still saved locally.")
else:
    print("\nTuned model did NOT beat the baseline LoRA run — keeping the baseline as Production.")
    print("Consider raising N_TRIALS for a wider search, or manually trying the runner-up configs "
          "in lora_hpt_leaderboard.csv.")

print(f"\nDownload these from the Kaggle notebook's Output tab before your session ends:")
print(f"  - {MODELS_DIR}/finbert_lora_tuned_adapter/")
print(f"  - {RESULTS_DIR}/lora_hpt_leaderboard.csv")
print(f"  - {os.path.join(BASE_DIR, 'mlflow.db')}")